# Topic 26 — Word Embeddings
### Theory → the limits of sparse vectors → train Word2Vec with gensim → explore similarity → torch embedding layer.

TF-IDF vectors (Topic 24) are **sparse** and **don't capture meaning** — "great" and "excellent"
get completely unrelated columns, even though they mean similar things. **Word embeddings** solve
this: every word becomes a **dense vector** (e.g. 100 real numbers, mostly non-zero), positioned in
space so that semantically similar words end up close together — a **distributional representation**
learned from the contexts words appear in ("you shall know a word by the company it keeps").

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

rng = np.random.default_rng(0)

## 1. Sparse vs dense — a quick contrast

- **TF-IDF vector**: length = vocabulary size (thousands), mostly zeros, no notion of similarity
  between different words.
- **Embedding vector**: length = a small fixed number (e.g. 50-300), all dimensions meaningful,
  distance between vectors reflects semantic similarity.

In [ ]:
tfidf_like = np.zeros(10000)
tfidf_like[42] = 0.7   # "great" happens to be vocabulary index 42

embedding_like = np.array([0.12, -0.45, 0.88, 0.03, -0.21])   # a dense, 5-dim made-up embedding

print("TF-IDF-style vector: length", len(tfidf_like), ", mostly zeros")
print("Embedding-style vector: length", len(embedding_like), ", every value is meaningful:", embedding_like)

## 2. Word2Vec: CBOW vs Skip-gram

Both train a small neural network on a "fill in the blank" style task using surrounding context
words, then keep the network's learned INTERNAL weights as the word vectors (the actual
prediction task is just a means to an end).

- **CBOW** (Continuous Bag of Words): given surrounding context words, predict the missing middle word.
  `["the", "___", "sat", "on"]` -> predict "cat"
- **Skip-gram**: the reverse — given one word, predict its surrounding context words.
  `"cat"` -> predict `["the", "sat", "on"]`

Skip-gram tends to work better on rare words; CBOW trains faster. `gensim`'s `Word2Vec` lets you
pick via `sg=0` (CBOW, default) or `sg=1` (skip-gram).

In [ ]:
# Toy corpus -- in practice you'd want thousands+ of sentences for decent embeddings.
# This is just enough to demonstrate the mechanics.
sentences = [
    "you are stupid and worthless".split(),
    "you are an idiot and a loser".split(),
    "i hate you so much".split(),
    "great job today team well done".split(),
    "nice work everyone excellent effort".split(),
    "have a wonderful and great day".split(),
    "thanks for your excellent help today".split(),
    "you should just disappear you loser".split(),
    "well done on the great project".split(),
    "everyone thinks you are pathetic and stupid".split(),
] * 20   # repeat to give the tiny toy model enough training signal

model_cbow = Word2Vec(sentences, vector_size=20, window=3, min_count=1, sg=0, epochs=50, seed=42)
model_skipgram = Word2Vec(sentences, vector_size=20, window=3, min_count=1, sg=1, epochs=50, seed=42)

print("vocabulary:", list(model_cbow.wv.key_to_index.keys()))
print("\nvector for 'stupid' (CBOW):", model_cbow.wv["stupid"][:5], "... (20 dims total)")

## 3. Semantic similarity between word vectors

Cosine similarity between two embedding vectors measures how "close" their meanings are
(1 = identical direction/meaning, 0 = unrelated, -1 = opposite).

In [ ]:
print("similarity('stupid', 'idiot'):", model_cbow.wv.similarity("stupid", "idiot"))
print("similarity('great', 'excellent'):", model_cbow.wv.similarity("great", "excellent"))
print("similarity('stupid', 'great'):", model_cbow.wv.similarity("stupid", "great"))

print("\nmost similar words to 'stupid':")
for word, score in model_cbow.wv.most_similar("stupid", topn=5):
    print(f"  {word:<12} {score:.3f}")
# With this TINY toy corpus the numbers won't be very meaningful -- real embeddings need much
# more training data to place semantically similar words genuinely close together.

## 4. Visualizing embeddings with PCA (reusing Topic 19)

Project the high-dimensional embeddings down to 2D so we can see roughly how words cluster.

In [ ]:
words = list(model_cbow.wv.key_to_index.keys())
vectors = np.array([model_cbow.wv[w] for w in words])

pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(vectors)

plt.figure(figsize=(9, 7))
plt.scatter(vectors_2d[:, 0], vectors_2d[:, 1], alpha=0.3)
for i, word in enumerate(words):
    plt.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]), fontsize=9)
plt.title("Word2Vec embeddings projected to 2D (toy corpus)")
plt.show()
# With more real training data, you'd expect to see toxic/negative words cluster together,
# and positive words cluster separately.

## 5. Turning a sentence into a single vector (averaging word embeddings)

A simple baseline way to get one fixed-size vector per SENTENCE/document from word-level
embeddings: average all its word vectors together.

In [ ]:
def sentence_to_avg_vector(sentence, model):
    words = sentence.lower().split()
    vectors = [model.wv[w] for w in words if w in model.wv]
    if not vectors:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

vec1 = sentence_to_avg_vector("you are stupid", model_cbow)
vec2 = sentence_to_avg_vector("great job today", model_cbow)
print("sentence vector shape:", vec1.shape)

from numpy.linalg import norm
cos_sim = np.dot(vec1, vec2) / (norm(vec1) * norm(vec2))
print("cosine similarity between the two sentence vectors:", cos_sim)
# This averaged-embedding approach is a common quick baseline before moving to
# LSTM/Transformer-based sentence representations (Topics 34, 37-38).

## 6. GloVe and FastText — brief comparison, no training needed here

- **Word2Vec**: trained purely from local context windows (as above).
- **GloVe**: trained on global word co-occurrence statistics across the WHOLE corpus, not just local windows.
- **FastText**: like Word2Vec, but represents each word as a bag of CHARACTER n-grams too — so it
  can generate a reasonable vector even for misspelled or unseen words (very relevant for noisy
  social-media/cyberbullying text with typos and slang).

You typically load pretrained GloVe/FastText vectors (trained on huge corpora like Wikipedia)
rather than training your own from scratch, especially for a project-sized dataset.

In [ ]:
# Loading pretrained vectors via gensim's downloader (uncomment to actually download -- can be large)
# import gensim.downloader as api
# glove_vectors = api.load("glove-wiki-gigaword-50")   # ~66MB, 50-dim GloVe vectors
# print(glove_vectors.most_similar("stupid", topn=5))

print("Pretrained embeddings (GloVe/FastText) are usually a stronger starting point than training")
print("your own Word2Vec from a small project dataset -- they've already learned from huge corpora.")

## Exercise

In [ ]:
# --- Try it yourself ---
# 1. Increase `sentences` repetition to 100x and re-check model_cbow.wv.most_similar("stupid") --
#    does it start looking more sensible with more training data?
# 2. Try sg=1 (skip-gram) with the same corpus and compare its most_similar results to CBOW's.
# 3. Use sentence_to_avg_vector on 3 of your own toxic-sounding vs neutral sentences, and compute
#    pairwise cosine similarities between all of them -- do toxic sentences end up closer together?
# 4. Look up (or load) pretrained GloVe vectors with gensim.downloader and check
#    glove_vectors.most_similar("stupid") -- compare the quality to your toy-trained model.

---
### Next up: **Topic 27 — Neural Network Fundamentals** (before CNN/RNN/Transformers).

Say "next" when you're ready.